# 15 — Decision Tree sul test ufficiale (T31)

**Una prova singola**, prima di decidere se estendere agli altri modelli:
training sul 100% di KDDTrain+, valutazione su KDDTest+.
Profondità 15, seed 1, nessun tuning. Non riutilizziamo un albero degli split
precedenti: ne addestriamo uno da zero con la configurazione congelata in T28.

Gli split precedenti 80/20 misuravano validation interna. Questo test contiene
anche attacchi mai osservati nel training: il confronto non isola il solo effetto
di avere più righe per addestrare. Il test è ora consultato; eventuali modifiche
guidate da questi risultati saranno esperimenti esplorativi.

Ogni esecuzione completa di questo notebook esegue **un fit del classificatore**.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import sys
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_recall_fscore_support,
)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src/data.py').is_file()
sys.path.insert(0, str(ROOT))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS
from src.targets import AUTHOR_ATTACK_TO_FAMILY, FAMILY_ORDER

OUTPUT = ROOT / 'artifacts/decision_tree_official_test'
OUTPUT.mkdir(parents=True, exist_ok=True)

## 1. Protocollo e dati

CSV senza intestazione nella cartella del progetto. Verifichiamo identità dei
file, versioni e schema prima dell'esperimento. Non creiamo un nuovo split.

In [ ]:
protocol_path = ROOT / 'configs/official_test_v1.json'
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
protocol_hash = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
assert protocol['label_profile'] == 'author_75b8f8b'
assert protocol['features'] == list(FEATURE_COLUMNS)
assert protocol['family_order'] == list(FAMILY_ORDER)
assert protocol['split'] is None and protocol['test_tuning_allowed'] is False
versions = {name: importlib.metadata.version(name) for name in protocol['versions']}
assert versions == protocol['versions'], 'Versioni diverse dal protocollo'
for relative_path, expected in protocol['input_sha256'].items():
    assert hashlib.sha256((ROOT / relative_path).read_bytes()).hexdigest() == expected

string_types = {name: 'string' for name in (*CATEGORICAL_COLUMNS, 'attack_label')}
train = pd.read_csv(ROOT / 'data/raw/KDDTrain+.csv', header=None, names=COLUMN_NAMES, dtype=string_types)
test = pd.read_csv(ROOT / 'data/raw/KDDTest+.csv', header=None, names=COLUMN_NAMES, dtype=string_types)
numeric_columns = [name for name in FEATURE_COLUMNS if name not in CATEGORICAL_COLUMNS]
for name, frame in [('train', train), ('test', test)]:
    assert len(frame) == protocol['expected_rows'][name]
    assert frame.columns.tolist() == list(COLUMN_NAMES)
    assert not frame.isna().any().any()
    assert np.isfinite(frame[numeric_columns + ['difficulty']].to_numpy()).all()

## 2. Feature e target

Solo le 41 feature entrano nel modello. Target: cinque famiglie secondo il codice
autore, con worm → R2L e httptunnel → U2R. Le vecchie valutazioni restano invariate.
`difficulty` e il nome dell'attacco non sono predittori.

In [ ]:
for name, frame in [('train', train), ('test', test)]:
    unknown = ~frame['attack_label'].isin(AUTHOR_ATTACK_TO_FAMILY)
    if unknown.any():
        raise ValueError(f"Etichette non mappate in {name}: {frame.loc[unknown, 'attack_label'].tolist()}")
y_train = train['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY).rename('family')
y_test = test['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY).rename('family')
X_train = train.loc[:, list(FEATURE_COLUMNS)].copy()
X_test = test.loc[:, list(FEATURE_COLUMNS)].copy()
display(pd.DataFrame({'training': y_train.value_counts(), 'test': y_test.value_counts()}).reindex(FAMILY_ORDER))

## 3. Preprocessing — apprendimento solo sul training

One-hot per protocol_type, service e flag; numeriche lasciate nella scala originale
come nei precedenti esperimenti con gli alberi. Le colonne aumentano per effetto
dell'one-hot, ma provengono sempre dalle stesse 41 feature.

In [ ]:
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), list(CATEGORICAL_COLUMNS)),
    ('numeric', 'passthrough', numeric_columns),
])
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
encoder = preprocessor.named_transformers_['categorical']
for column, categories in zip(CATEGORICAL_COLUMNS, encoder.categories_):
    assert set(categories) == set(X_train[column])
assert X_train_encoded.shape[0] == len(train)
assert X_train_encoded.shape[1] == X_test_encoded.shape[1]
assert np.isfinite(X_train_encoded).all() and np.isfinite(X_test_encoded).all()
print(f'Training: {X_train_encoded.shape}; test: {X_test_encoded.shape}')

## 4. Configurazione del modello

API sklearn originale, profondità massima 15 e seed 1. Verifichiamo anche tutti
gli altri parametri contro il protocollo congelato prima di chiamare `fit`.

In [ ]:
model = DecisionTreeClassifier(max_depth=15, random_state=1)
assert model.get_params() == protocol['models']['decision_tree']['parameters'], 'configurazione diversa dal protocollo' 

## 5. Training — un solo fit sul 100% di KDDTrain+

Nessun target del test viene passato a `fit`. Il seed rende ripetibile questa
prova, ma una prova singola non misura la variabilità tra inizializzazioni.

In [ ]:
fit_start = perf_counter()
model.fit(X_train_encoded, y_train)
fit_seconds = perf_counter() - fit_start
print(f'Righe usate: {len(y_train):,}; profondità: {model.get_depth()}; foglie: {model.get_n_leaves()}')
print(f'Tempo fit: {fit_seconds:.2f} secondi')

## 6. Predizione sul test ufficiale

Applichiamo il modello già addestrato. Non scegliamo parametri in base all'esito.

In [ ]:
predict_start = perf_counter()
y_pred = model.predict(X_test_encoded)
predict_seconds = perf_counter() - predict_start
assert len(y_pred) == len(y_test)

## 7. Metriche globali e per famiglia

Macro F1 assegna lo stesso peso alle cinque famiglie; weighted F1 pesa il supporto
ed è la metrica aggregata usata nel confronto con il paper. `error_rate` è 1−recall:
la quota di esempi di una classe assegnati altrove. Precision e falsi positivi
servono per leggere anche gli allarmi sbagliati.

In [ ]:
cm_counts = confusion_matrix(y_test, y_pred, labels=FAMILY_ORDER)
cm_normalized = confusion_matrix(y_test, y_pred, labels=FAMILY_ORDER, normalize='true')
precision, recall, class_f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=FAMILY_ORDER, zero_division=0,
)
class_metrics = pd.DataFrame({
    'family': FAMILY_ORDER,
    'precision': precision,
    'recall': recall,
    'f1': class_f1,
    'support': support,
    'true_positives': np.diag(cm_counts),
    'false_positives': cm_counts.sum(axis=0) - np.diag(cm_counts),
    'false_negatives': cm_counts.sum(axis=1) - np.diag(cm_counts),
    'error_rate': 1 - recall,
})
metrics = {
    'macro_f1': float(f1_score(y_test, y_pred, labels=FAMILY_ORDER, average='macro', zero_division=0)),
    'weighted_f1': float(f1_score(y_test, y_pred, labels=FAMILY_ORDER, average='weighted', zero_division=0)),
    'accuracy': float(accuracy_score(y_test, y_pred)),
}

In [ ]:
display(pd.Series(metrics, name='test ufficiale').to_frame().round(4))
display(class_metrics.round(4))

## 8. Confusion matrix

Righe = famiglia vera, colonne = famiglia predetta. A destra ogni riga somma a 1:
la diagonale è il recall della famiglia. A sinistra vediamo i conteggi effettivi,
necessari per valutare il peso degli errori sulle classi rare.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm_counts, display_labels=FAMILY_ORDER).plot(ax=axes[0], cmap='Blues', values_format='d', colorbar=False)
ConfusionMatrixDisplay(cm_normalized, display_labels=FAMILY_ORDER).plot(ax=axes[1], cmap='Blues', values_format='.3f', colorbar=False)
axes[0].set_title('Test ufficiale — conteggi')
axes[1].set_title('Test ufficiale — normalizzata per classe vera')
for axis in axes:
    axis.set_xlabel('Famiglia predetta')
    axis.set_ylabel('Famiglia vera')
fig.tight_layout()
fig.savefig(OUTPUT / 'confusion_matrices.png', dpi=160, bbox_inches='tight')
plt.show()

## 9. Visualizzazione dei primi livelli

Mostriamo solo i primi tre livelli del **medesimo albero di profondità 15**.
Il grafico è troncato per leggibilità: nessun albero più corto viene addestrato
e tutte le metriche derivano dall'albero completo.

In [ ]:
fig, axis = plt.subplots(figsize=(20, 8))
plot_tree(model, max_depth=2, feature_names=preprocessor.get_feature_names_out(),
          class_names=model.classes_.tolist(), filled=True, rounded=True,
          impurity=False, proportion=True, fontsize=8, ax=axis)
axis.set_title('Primi tre livelli del Decision Tree completo (profondità 15)')
fig.tight_layout()
fig.savefig(OUTPUT / 'tree_top_levels.png', dpi=130, bbox_inches='tight')
plt.show()

## 10. Salvataggio delle evidenze

Cartella separata dagli esperimenti precedenti. Salviamo predizioni in ordine
originale, indici, target, parametri, versioni e hash del protocollo. Si possono
ricontrollare tutte le metriche senza rifare il training.

In [ ]:
np.savez_compressed(
    OUTPUT / 'predictions.npz',
    test_indices=test.index.to_numpy(),
    truth=y_test.to_numpy(dtype=str),
    predicted=np.asarray(y_pred, dtype=str),
)
class_metrics.to_csv(OUTPUT / 'class_metrics.csv', index=False)
pd.DataFrame(cm_counts, index=FAMILY_ORDER, columns=FAMILY_ORDER).to_csv(OUTPUT / 'confusion_counts.csv', index_label='true_family')
pd.DataFrame(cm_normalized, index=FAMILY_ORDER, columns=FAMILY_ORDER).to_csv(OUTPUT / 'confusion_normalized.csv', index_label='true_family')
results = {
    'schema_version': 1,
    'model_id': 'decision_tree_official_test',
    'evaluation_set': 'KDDTest+ full, single pilot run',
    'protocol_id': protocol['protocol_id'],
    'protocol_sha256': protocol_hash,
    'input_sha256': protocol['input_sha256'],
    'label_profile': protocol['label_profile'],
    'family_order': list(FAMILY_ORDER),
    'train_rows': len(y_train), 'test_rows': len(y_test),
    'raw_feature_count': len(FEATURE_COLUMNS),
    'encoded_feature_count': X_train_encoded.shape[1],
    'classifier_parameters': model.get_params(),
    'model_fits': 1,
    'seed': 1,
    'versions': versions,
    'preprocessing': 'one-hot train-only; numeric passthrough',
    'encoder_categories': [values.tolist() for values in encoder.categories_],
    'numeric_columns': numeric_columns,
    'fit_seconds': fit_seconds, 'predict_seconds': predict_seconds,
    'actual_depth': int(model.get_depth()), 'leaves': int(model.get_n_leaves()),
    'metrics': metrics,
    'class_metrics': class_metrics.to_dict(orient='records'),
    'confusion_counts': cm_counts.tolist(),
    'confusion_normalized': cm_normalized.tolist(),
    'predictions_sha256': hashlib.sha256((OUTPUT / 'predictions.npz').read_bytes()).hexdigest(),
    'test_consulted': True,
}
(OUTPUT / 'results.json').write_text(json.dumps(results, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Evidenze salvate. Nessun altro modello eseguito.')

## 11. Come interpretare questa prova

Una singola valutazione sul test ufficiale, non una media di cinque split.
Le classi del test e la loro distribuzione differiscono dalla validation interna;
in particolare il profilo autore assegna 200 esempi a U2R. Non attribuiamo un
eventuale peggioramento al solo passaggio dall'80% al 100% del training.

Restano 654 vettori di feature distinti condivisi tra train e test, conservati dal
protocollo: il test non è composto interamente da esempi nuovi. I dettagli sono
nel notebook 14. Non deduciamo superiorità o inferiorità rispetto al paper da
configurazioni diverse. Prima di altri esperimenti leggiamo confusion matrix,
precision e supporti, senza modificare automaticamente questo modello.

## Risultato misurato — 5 novembre 2026

Macro F1 **0.5099**, weighted F1 **0.7186**,
accuracy **75.96%**.

| Famiglia | Precision | Recall | Errore (1−recall) | Corretti/totale |
|---|---:|---:|---:|---:|
| Normal | 65.98% | 97.34% | 2.66% | 9453/9711 |
| DoS | 96.03% | 80.42% | 19.58% | 5998/7458 |
| Probe | 83.99% | 62.62% | 37.38% | 1516/2421 |
| R2L | 95.54% | 5.45% | 94.55% | 150/2754 |
| U2R | 87.50% | 3.50% | 96.50% | 7/200 |

Le difficoltà maggiori sono R2L e U2R: rispettivamente 2590 e 181 esempi vengono
classificati come Normal. La precision elevata delle poche predizioni R2L/U2R non
compensa il basso recall: la maggior parte di questi attacchi sfugge al modello.

Il precedente Decision Tree15 aveva macro F1 medio **0,8754** nei cinque holdout
interni; qui ottiene **0,5099** sul test ufficiale. Weighted F1 passa da **0,9973**
a **0,7186**; recall R2L da **93,47%** a **5,45%**, U2R da **40%** a **3,50%**.
Non sono prove sullo stesso insieme: cambiano campioni, distribuzione, attacchi
presenti, quantità di training e, nel test, il profilo di due label. Quindi non
possiamo attribuire il divario all'uso del 100% del training né misurare da questo
confronto il contributo di ciascuna causa.

La prova mostra un limite marcato su questo test. Non cambiamo automaticamente
profondità, feature o altri parametri. Gli altri cinque modelli restano da eseguire
solo dopo decisione dell'utente. Il test ufficiale è ora consultato: sviluppi
guidati dai suoi risultati andranno dichiarati esplorativi.

**Nota tecnica:** il salvataggio JSON è stato corretto e la stessa configurazione
rieseguita una volta; predizioni identiche. Una configurazione, due fit effettivi,
nessuna media. Dettagli in docs/OFFICIAL_DECISION_TREE_RESULTS.md.
